In [1]:
import h5py
import torch
import os
import numpy as np
from src.models import BinaryClassificationModel

from trident import OpenSlideWSI, visualize_heatmap

In [3]:
#  e. Run inference
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
slide_id = "18-I-9488"
artifacts_dir = "./artifacts_max"
job_dir = "./heatmaps"
fold_num = 0
trial_num = 59
patch_features_path = f"features_conch_v15/{slide_id}.h5"
slide_path = f"D:\\2025_project_nsclc_unimi\\original_slides\\{slide_id}.ndpi"


# Model loading
model_path = os.path.join(artifacts_dir, f"trial_{trial_num}", "models", f"fold_{fold_num}.pt")
model = BinaryClassificationModel(n_heads=2, hidden_dim=256)
model.load_state_dict(torch.load(model_path, map_location=torch.device(device)))
model.to(device)


# slide loading and patching
slide = OpenSlideWSI(slide_path, lazy_init=False)
with h5py.File(patch_features_path, 'r') as f:
    coords = f['coords'][:]
    patch_features = f['features'][:]
    coords_attrs = dict(f['coords'].attrs)

batch = {'features': torch.from_numpy(patch_features).float().to(device).unsqueeze(0)}
model.eval()
with torch.no_grad(): # Saves memory during inference
    logits, attention = model(batch, return_raw_attention=True)

# Extract the numpy array
raw_scores = attention.cpu().numpy().squeeze()

# Reduce to 1D if we have multiple heads
if raw_scores.ndim >= 2:
    # Check which axis represents the heads (the axis with size 2)
    head_axis = 0 if raw_scores.shape[0] == 2 else 1

    # Option A: Average the attention scores across the heads (Recommended)
    scores_1d = np.mean(raw_scores, axis=head_axis)

    # Option B: Or, just select the first head
    # scores_1d = raw_scores[0, :] if head_axis == 0 else raw_scores[:, 0]
else:
    scores_1d = raw_scores

# f. generate heatmap
heatmap_save_path = visualize_heatmap(
    wsi=slide,
    scores=scores_1d, # Pass the corrected 1D array here
    coords=coords,
    vis_level=2,
    vis_mag=20,
    patch_size_level0=coords_attrs['patch_size_level0'],
    normalize=True,
    num_top_patches_to_save=10,
    output_dir=job_dir,
    filename=slide_id,
)

# # f. generate heatmap
# heatmap_save_path = visualize_heatmap(
#     wsi=slide,
#     scores=attention.cpu().numpy().squeeze(),
#     coords=coords,
#     vis_level=1,
#     patch_size_level0=coords_attrs['patch_size_level0'],
#     normalize=True,
#     num_top_patches_to_save=10,
#     output_dir=job_dir
# )